In [2]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from PIL import Image
import numpy as np
import json

PROJET = Path("/content/drive/MyDrive/MiniProjet-Rétinographie")
BASE = PROJET / "dataset" / "DRIVE"
PROCESSED = PROJET / "dataset" / "processed"

PROCESSED.mkdir(parents=True, exist_ok=True)

TAILLE = 512
IDS_VALIDATION = ["36", "30", "35", "28"]

EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".gif", ".bmp"
}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
def indexer(dossier):
    if not dossier.is_dir():
        raise FileNotFoundError(dossier)

    index = {}

    for chemin in sorted(dossier.iterdir()):
        if chemin.is_file() and chemin.suffix.lower() in EXTENSIONS:
            identifiant = chemin.stem.split("_")[0]

            if identifiant in index:
                raise ValueError(f"Identifiant en double : {identifiant}")

            index[identifiant] = chemin

    if not index:
        raise ValueError(f"Aucune image dans {dossier}")

    return index


photos = indexer(BASE / "training/images")
annotations = indexer(BASE / "training/1st_manual")
champs_train = indexer(BASE / "training/mask")

photos_test = indexer(BASE / "test/images")
champs_test = indexer(BASE / "test/mask")

assert set(photos) == set(annotations) == set(champs_train), (
    "Correspondances incomplètes dans training."
)
assert set(photos_test) == set(champs_test), (
    "Correspondances incomplètes dans test."
)
assert set(IDS_VALIDATION).issubset(photos), (
    "Une image de validation est absente."
)

# Reproduire l'ordre utilisé lors de l'entraînement précédent
ids_ordonnes = sorted(photos)
ordre = np.random.default_rng(42).permutation(len(ids_ordonnes))

ids_val = [ids_ordonnes[i] for i in ordre[:4]]
ids_train = [ids_ordonnes[i] for i in ordre[4:]]
ids_test = sorted(photos_test)

assert ids_val == IDS_VALIDATION, (
    "La séparation ne correspond plus à l'expérience précédente."
)

print("Entraînement :", len(ids_train), "images")
print("Validation :", ids_val)
print("Test :", len(ids_test), "images")

Entraînement : 16 images
Validation : ['36', '30', '35', '28']
Test : 20 images


In [4]:
def lire_photo(chemin):
    with Image.open(chemin) as fichier:
        image = fichier.convert("RGB").resize(
            (TAILLE, TAILLE),
            Image.Resampling.BILINEAR
        )
        return np.asarray(image, dtype=np.float32) / 255.0


def lire_masque(chemin):
    with Image.open(chemin) as fichier:
        masque = fichier.convert("L").resize(
            (TAILLE, TAILLE),
            Image.Resampling.NEAREST
        )
        return (np.asarray(masque) > 127).astype(np.uint8)[..., None]


def charger_groupe(ids):
    # Vérifier les dimensions avant redimensionnement
    for identifiant in ids:
        tailles = []

        for chemin in [
            photos[identifiant],
            annotations[identifiant],
            champs_train[identifiant]
        ]:
            with Image.open(chemin) as fichier:
                tailles.append(fichier.size)

        if len(set(tailles)) != 1:
            raise ValueError(
                f"Dimensions incompatibles pour l'image {identifiant}"
            )

    X = np.stack([lire_photo(photos[i]) for i in ids])
    Y = np.stack([lire_masque(annotations[i]) for i in ids])
    FOV = np.stack([lire_masque(champs_train[i]) for i in ids])

    return X, Y, FOV


X_train, Y_train, FOV_train = charger_groupe(ids_train)
X_val, Y_val, FOV_val = charger_groupe(ids_val)

for identifiant in ids_test:
    with Image.open(photos_test[identifiant]) as photo:
        with Image.open(champs_test[identifiant]) as champ:
            if photo.size != champ.size:
                raise ValueError(
                    f"Dimensions incompatibles pour le test {identifiant}"
                )

X_test = np.stack([lire_photo(photos_test[i]) for i in ids_test])
FOV_test = np.stack([lire_masque(champs_test[i]) for i in ids_test])

print("X_train :", X_train.shape)
print("Y_train :", Y_train.shape)
print("X_val   :", X_val.shape)
print("Y_val   :", Y_val.shape)
print("X_test  :", X_test.shape)

X_train : (16, 512, 512, 3)
Y_train : (16, 512, 512, 1)
X_val   : (4, 512, 512, 3)
Y_val   : (4, 512, 512, 1)
X_test  : (20, 512, 512, 3)


In [6]:
np.savez_compressed(
    PROCESSED / "train.npz",
    X=X_train,
    Y=Y_train,
    FOV=FOV_train
)

np.savez_compressed(
    PROCESSED / "validation.npz",
    X=X_val,
    Y=Y_val,
    FOV=FOV_val
)

np.savez_compressed(
    PROCESSED / "test.npz",
    X=X_test,
    FOV=FOV_test
)

description = {
    "taille": TAILLE,
    "seed": 42,
    "normalisation": "RGB float32 entre 0 et 1",
    "masques": "0 = fond ; 1 = vaisseau",
    "train": [photos[i].name for i in ids_train],
    "validation": [photos[i].name for i in ids_val],
    "test": [photos_test[i].name for i in ids_test],
    "annotations_test_disponibles": False
}

with open(PROCESSED / "preparation.json", "w", encoding="utf-8") as fichier:
    json.dump(description, fichier, indent=2, ensure_ascii=False)

# Vérifier que les fichiers peuvent être relus
for nom in ["train.npz", "validation.npz", "test.npz"]:
    with np.load(PROCESSED / nom) as donnees:
        print(nom, "→", {
            cle: donnees[cle].shape for cle in donnees.files
        })

print("\nDonnées préparées sauvegardées dans :", PROCESSED)

train.npz → {'X': (16, 512, 512, 3), 'Y': (16, 512, 512, 1), 'FOV': (16, 512, 512, 1)}
validation.npz → {'X': (4, 512, 512, 3), 'Y': (4, 512, 512, 1), 'FOV': (4, 512, 512, 1)}
test.npz → {'X': (20, 512, 512, 3), 'FOV': (20, 512, 512, 1)}

Données préparées sauvegardées dans : /content/drive/MyDrive/MiniProjet-Rétinographie/dataset/processed
